In [0]:
from pyspark.sql.functions import col, row_number, round, to_timestamp
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import timedelta

In [0]:
silver_table_name = "magnificent7stocks.02_silver_layer"

silver_df = spark.read.table(silver_table_name)

# Get the most recent ingestion_ts_utc
latest_ts = (silver_df
             .selectExpr("max(ingestion_ts_utc) as latest_ingestion_ts_utc")
             .collect()[0]["latest_ingestion_ts_utc"])
window_start = latest_ts - timedelta(days=2)

print("Latest ingestion timestamp:", latest_ts)
print("Window start:", window_start)

In [0]:
#----------------------------------------------
# 1. Read Bronze Layer
# ----------------------------------------------
bronze_df = spark.read.table("magnificent7stocks.01_bronze_layer")

bronze_df = bronze_df.filter(col("ingestion_ts_utc") > window_start)

display(bronze_df)

In [0]:
# ----------------------------------------------
# 2. Deduplicate using latest ingestion_ts_utc
# ----------------------------------------------
window_spec = (
    Window
    .partitionBy("datetime", "ticker")
    .orderBy(col("ingestion_ts_utc").desc())
)

deduped_df = (
    bronze_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
    .withColumn("datetime", to_timestamp("datetime"))
    .withColumn("adj_close", round(col("adj_close"), 2))
    .withColumn("high", round(col("high"), 2))
    .withColumn("low", round(col("low"), 2))
    .withColumn("open", round(col("open"), 2))     
    .withColumn("close", round(col("close"), 2))          
    .withColumn("volume", round(col("volume"), 2))
    .select("datetime", "ticker","adj_close", "high", "low", "open", "close", "volume", "ingestion_ts_utc")
)

# ----------------------------------------------
# 3. MERGE into Silver Delta Table
# ----------------------------------------------
silver_table_name = "magnificent7stocks.02_silver_layer"

if spark.catalog.tableExists(silver_table_name):
    print("Merging into existing Silver table...")

    silver_table = DeltaTable.forName(spark, silver_table_name)

    (
        silver_table.alias("t")
        .merge(
            deduped_df.alias("s"),
            "t.datetime = s.datetime AND t.ticker = s.ticker"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    print("Silver table does not exist — creating it...")
    (
        deduped_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table_name)
    )

print("Silver layer update complete.")